<a href="https://colab.research.google.com/github/dcpetty/google-colaboratory/blob/main/iso/iso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# google-colaboratory/iso

This is a [Google Colab](https://colab.research.google.com/) notebook that downloads and parses ISO and UNSD standards data for language and country codes. Understanding these [Byzantine](https://en.wikipedia.org/wiki/Byzantine_(disambiguation)) codes is part of legitimizing use of the [MIT App Inventor](https://appinventor.mit.edu) [TextToSpeech](https://ai2.appinventor.mit.edu/reference/components/media.html#TextToSpeech) and [SpeechRecognizer](https://ai2.appinventor.mit.edu/reference/components/media.html#SpeechRecognizer) components.

The use of the [`iso-data.json`](https://github.com/dcpetty/google-colaboratory/blob/main/iso/iso-data.json) is demonstrated with the [MIT App Inventor](https://appinventor.mit.edu) [Languages](https://dcpetty.dev/mit-app-inventor/Languages/) app.

## ISO 639

- [https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes](https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes) ISO 639 language codes. 'ISO 639 is a standardized nomenclature used to classify languages. Each language is assigned a two-letter&hellip; and three-letter lowercase abbreviation&hellip;'
- [http://download.geonames.org/export/dump/iso-languagecodes.txt](http://download.geonames.org/export/dump/iso-languagecodes.txt) Raw ISO 639 language codes.
- Note: [http://download.geonames.org/export/dump/alternateNames.zip](http://download.geonames.org/export/dump/alternateNames.zip) contains `iso-languagecodes.txt`, as well as the &approx;730MB file `alternateNames.txt` with *a lot* more detail.

## ISO 3166

- [https://en.wikipedia.org/wiki/ISO_3166](https://en.wikipedia.org/wiki/ISO_3166) ISO 3166 *codes for the representation of names of countries and their subdivisions*. 'ISO 3166 is a standard published by the International Organization for Standardization (ISO) that defines codes for the names of countries, dependent territories, special areas of geographical interest, and their principal subdivisions (*e.g.*, provinces or states).'
- [http://download.geonames.org/export/dump/countryInfo.txt](http://download.geonames.org/export/dump/countryInfo.txt) Raw ISO 3166 region codes.

## UNSD M49

- [https://unstats.un.org/unsd/methodology/m49/](https://unstats.un.org/unsd/methodology/m49/) Statistics Division of the United Nations Secretariat (UNSD) M49 *Standard Country or Area Codes for Statistical Use*.
- [https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv](https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv) ISO 3166 list listing all *regional*, *sub-regional*, and *intermediate-regional* codes.
- Also, [`2022-09-24__CSV_UNSD_M49.csv`](https://raw.githubusercontent.com/omnika-datastore/unsd-m49-standard-area-codes/main/2022-09-24__CSV_UNSD_M49.csv) includes these data.

## `iso-data.json`

The generated `iso-data.json` includes tagged data for `"country_lookup"` mapped by country and `"language_lookup"` mapped by language. For example:

```
{
  "_comment": // ...
  "country_lookup": {
    // ...
    "FRA": {
      "iso2": "FR",
      "name": "France",
      "region": {
        "iso2": "EU",
        "name": "Europe",
        "code": "150"
      },
      "languages": [
        {
          "code": "fr-FR",
          "name": "French"
        },
        {
          "code": "frp",
          "name": "Arpitan"
        },
        {
          "code": "br",
          "name": "Breton"
        },
        {
          "code": "co",
          "name": "Corsican"
        },
        {
          "code": "ca",
          "name": "Catalan"
        },
        {
          "code": "eu",
          "name": "Basque"
        },
        {
          "code": "oc",
          "name": "Occitan (post 1500)"
        }
      ]
    },
    // ...
  }
  "language_lookup": {
    // ...
    "fr-FR": {
      "name": "French",
      "primary": [
        "FRA"
      ],
      "spoken": [
        "FRA"
      ]
    },
    // ...
  }
}
```

The last section of this notebook implements `valid_countries_for_language` and `valid_languages_for_country` (and `find_language_name` for valid country-specific languages that may not exist in the `"language_lookup"` list). Any [MIT App Inventor](https://appinventor.mit.edu) purporting to to check for valid landuages and valid countries will have to replicate those algorithms.

The downloading, parsing, and tagging functions were developed with the help of [Google Gemini](https://gemini.google.com/).

<hr>

Use [https://nbviewer.jupyter.org/github/dcpetty/google-colaboratory/blob/main/iso/iso.ipynb](https://nbviewer.jupyter.org/github/dcpetty/google-colaboratory/blob/main/iso/iso.ipynb) to view it in `nbviewer`.

In [3]:
#!/usr/bin/env python3
#
# Mount Google Drive, download remote files, and set local values.
#
# https://github.com/dcpetty/google-colaboratory/blob/main/remote-files/remote-files.ipynb
#
from os.path import join, realpath
dot_path = realpath('.')
gdrive = join(dot_path, 'gdrive')
notebooks = join(gdrive, 'My Drive/Colab Notebooks')
repo = 'iso'    # repo path within the Drive/Colab Notebooks directory
drive_path = join(notebooks, repo)
print(dot_path)
# https://colab.research.google.com/github/AllenDowney/ThinkPython/blob/v3/chapters/chap04.ipynb
from os.path import basename, exists
def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

# Mount Google Drive for accessing files in the same direcory as this notebook.
# https://colab.research.google.com/notebooks/io.ipynb#scrollTo=u22w3BFiOveA
from google.colab import drive
drive.mount(gdrive)
# %cd drive_path   # cd so . is repo_path

# Download remote files for accessing in the runtime directory of this notebook.
download('http://download.geonames.org/export/dump/iso-languagecodes.txt')
# download('http://download.geonames.org/export/dump/alternateNames.zip')
download('http://download.geonames.org/export/dump/countryInfo.txt')
download('https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv')

# Set global variables
verbose = False # whether to print data

/content
Mounted at /content/gdrive


In [4]:
#!/usr/bin/env python3
import csv
import json
import os
import os.path
import re
import ssl

def get_language_map(input_file, output_path="./iso-639-languages.json"):
    """Saves a JSON mapping from three columns of input_file to name (fourth
    column) and returns the dict."""
    lang_map = {}

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('ISO') or not line.strip():
                    continue
                fields = line.split('\t')
                if len(fields) >= 4:
                    name = fields[3].strip()
                    # Map Alpha-3, Alpha-2, and Alpha-1 to the same name
                    for i in range(3):
                        code = fields[i].strip()
                        if code:
                            lang_map[code] = name

        with open(output_path, 'w', encoding='utf-8') as jf:
            json.dump(lang_map, jf, indent=2, ensure_ascii=False)

        print(f"Created {os.path.basename(output_path)} with {len(lang_map)} language names")

        return lang_map

    except Exception as e:
        print(f"Error processing languages TXT: {e}")
        return {}

def get_country_list(input_file, output_path="./iso-3166-countries.json"):
    """Saves the original country list to JSON, and returns the list."""
    country_list = []

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                fields = line.split('\t')
                # TLD is field 9; Languages is field 15
                if len(fields) > 15:
                    entry = {
                        "iso2": fields[0],
                        "iso3": fields[1],
                        "numeric_m49": fields[2],
                        "country": fields[4],
                        "continent": fields[8],
                        "tld": fields[9],
                        "languages": [l.strip() for l in fields[15].split(',')] if fields[15] else []
                    }
                    country_list.append(entry)

        with open(output_path, 'w', encoding='utf-8') as jf:
            json.dump(country_list, jf, indent=2, ensure_ascii=False)

        print(f"Created {os.path.basename(output_path)} with {len(country_list)} countries")

        return country_list

    except Exception as e:
        print(f"Error processing countries TXT: {e}")
        return []

def download_github_files(url, local_path):
    """Robust downloader that handles the SSL certificate issue.
    NOTE: not used in this notebook."""
    context = ssl._create_unverified_context()

    try:
        from urllib.request import urlopen
        with urlopen(url, context=context) as response:
            with open(local_path, 'wb') as f:
                f.write(response.read())
        return True

    except Exception as e:
        print(f"Download failed: {e}")
        return False

def generate_3166_continents_map(categorization):
    """
    Incorporate GeoNames continent short-names (AF, EU, etc.) into
    the M49 categorization structure.
    """
    # http://download.geonames.org/export/dump/countryInfo.txt
    # Map of GeoNames continent codes from countryInfo.txt to UN M49 names.
    continent_names = {
        "AF": "Africa",
        "AN": "Antarctica",
        "AS": "Asia",
        "EU": "Europe",
        "NA": "Northern America",
        "OC": "Oceania",
        "SA": "South America"
    }

    # Create name -> code lookups for efficient cross-referencing
    # We combine all levels because GeoNames continents can map to
    # Regions (Africa) or Intermediate Regions (South America)
    continents_map = {}
    regions_name_map = {d['name']: d['code'] for d in categorization["regions"]}
    sub_regions_name_map = {d['name']: d['code'] for d in categorization["sub_regions"]}
    intermediate_name_map = {d['name']: d['code'] for d in categorization["intermediate_regions"]}

    for t_iso2, t_name in continent_names.items():
        t_code = None

        # Check all three tiers for the matching name to find the M49 code
        if t_name in regions_name_map:
            t_code = regions_name_map[t_name]
        elif t_name in sub_regions_name_map:
            t_code = sub_regions_name_map[t_name]
        elif t_name in intermediate_name_map:
            t_code = intermediate_name_map[t_name]

        if t_name and t_code:
            continents_map[t_code] = {
                "iso2": t_iso2,
                "name": t_name,
                "code": t_code
            }

    # The all.csv file does not include M49 continent information for Antartica.
    continents_map['010'] = { 'iso2': 'AN', 'name': 'Antartica', 'code': '010', }

    return continents_map

def get_m49_categorization(input_file, output_path="./unsd-m49-categories.json"):
    """
    Downloads and parses full 3166 CSV to extract unique M49 regions, sub-regions
    and intermediate-regions as lists of { "name": ..., "code": ... } dicts,
    sorted by code.
    """
    # Using dictionaries to ensure uniqueness by 'code'
    regions_map = {}
    sub_regions_map = {}
    intermediate_regions_map = {}
    continents_map = {}

    try:
        with open(input_file, mode='r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f, delimiter=',')

            for row in reader:
                # Headers in Lukes CSV for intermediate regions:
                # 'intermediate-region', 'intermediate-region-code'
                r_name, r_code = row.get('region', ''), row.get('region-code', '')
                s_name, s_code = row.get('sub-region', ''), row.get('sub-region-code', '')
                i_name, i_code = row.get('intermediate-region', ''), row.get('intermediate-region-code', '')

                # check is an assertion check for duplicate codes
                def check(l, m, c, n): assert c not in m or m[c]['name'] == n, \
                    f"Duplicate {l} code: {c} ({m[c]['name']} & {n})"

                if r_name and r_code:
                    check('region', regions_map, r_code, r_name)
                    regions_map[r_code] = {"name": r_name, "code": r_code}
                if s_name and s_code:
                    check('sub-region', sub_regions_map, s_code, s_name)
                    sub_regions_map[s_code] = {"name": s_name, "code": s_code}
                if i_name and i_code:
                    check('intermediate-region', intermediate_regions_map, i_code, i_name)
                    intermediate_regions_map[i_code] = {"name": i_name, "code": i_code}

        categorization = {
            "regions": sorted(regions_map.values(), key=lambda x: x['code']),
            "sub_regions": sorted(sub_regions_map.values(), key=lambda x: x['code']),
            "intermediate_regions": sorted(intermediate_regions_map.values(), key=lambda x: x['code'])
        }

        continents_map = generate_3166_continents_map(categorization)
        categorization["continents_3166"] = sorted(continents_map.values(), key=lambda x: x['code'])

        with open(output_path, 'w', encoding='utf-8') as jf:
            json.dump(categorization, jf, indent=2, ensure_ascii=False)

        print(f"Created {os.path.basename(output_path)} with "
            f"{len(regions_map)} regions, "
            f"{len(sub_regions_map)} sub-regions, "
            f"{len(intermediate_regions_map)} intermediate-regions.")

        return categorization

    except Exception as e:
        print(f"Error processing M49 CSV: {e}")
        return {"regions": [], "sub_regions": [], "intermediate_regions": []}

def generate_tagged_mappings(country_list, lang_map, region_map, output_path="./iso-data.json"):
    """Creates the cross-referenced mappings and saves to JSON."""
    country_lookup = {}
    language_lookup = {}

    for entry in country_list:
        iso3 = entry['iso3']
        langs = entry['languages']

        # 1. Build Country Lookup (Alpha-3 key)
        country_lookup[iso3] = {
            "iso2": entry['iso2'],
            "name": entry['country'],
            # Add 'continent' dictionary from region_map
            "region": region_map.get(entry['continent'], dict()),
            # Enrich 'languages' with full names from the lang_map
            "languages": [{
                "code": l,
                "name": lang_map.get(re.sub(r'([^-]*)-.*', r'\1', l), "UNKNOWN")
            } for l in langs],
            # Add 'speak' to simplify 'languages'
            "speak": langs,
        }

        # 2. Build Language Lookup (Language code key)
        for index, lang_code in enumerate(langs):
            if not lang_code:
                continue

            if lang_code not in language_lookup:
                mod_lang_code = re.sub(r'([^-]*)-.*', r'\1', lang_code)
                language_lookup[lang_code] = {
                    "name": lang_map.get(re.sub(r'([^-]*)-.*', r'\1', lang_code), "Unknown"),
                    "primary": [],
                    "spoken": []
                }

            language_lookup[lang_code]["spoken"].append(iso3)
            if index == 0:
                language_lookup[lang_code]["primary"].append(iso3)

    _comment = \
"""
This file includes analysis of ISO 639 language codes, ISO 3166 'codes for the
representation of names of countries and their subdivisions,' and UNSD M49
'Standard Country or Area Codes for Statistical Use.'
""".replace('\n', ' ').strip()
    final_map = {
        "_comment": _comment,
        "country_lookup": country_lookup,
        "language_lookup": language_lookup
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_map, f, indent=2, ensure_ascii=False)

    print(f"Created {os.path.basename(output_path)} with "
        f"{len(country_lookup)} country keys, "
        f"{len(language_lookup)} language keys.")

    return final_map

if __name__ == "__main__":
    np = lambda name: join(drive_path, name)

    # 1. Get Language Names
    languages_dict = get_language_map('iso-languagecodes.txt', np('iso-639-languages.json'))

    # 2. Get Country List
    countries_data = get_country_list('countryInfo.txt', np('iso-3166-countries.json'))

    # 3. Generate M49 Categories
    m49_cats = get_m49_categorization('all.csv', np('unsd-m49-categories.json'))
    region_map = {region['iso2']: region for region in m49_cats['continents_3166']}

    # 4. Generate Tagged Mappings
    tagged_data = generate_tagged_mappings(countries_data, languages_dict, region_map,
        np('iso-data.json'))

    # Debug / Verification
    print(f"Processed {len(countries_data)} countries and {len(languages_dict)} language names.")

    verbose, number = False, 20
    if not verbose:

        print(f"\nlanguages_dict")
        for language in [key for key in sorted(languages_dict) if len(key) == 2][: number]:
            print(f"{language}: {languages_dict[language]}")

        print(f"\ncountries_data")
        for country in countries_data[: number]:
            print(f"{country}")

        for key in m49_cats:
            print(f"\nm49_cats['{key}']")
            for value in m49_cats[key][: number]:
                print(f"{value}")

        for tag in tagged_data:
            if not tag.startswith('_'):
                print(f"\ntagged_data['{tag}']")
                for key in sorted(tagged_data[tag])[: number]:
                    print(f"{key}: {tagged_data[tag][key]}")

    # Optional: Quick check on a specific language (e.g., 'es' for Spanish)
    if 'es' in tagged_data['language_lookup']:
        es_info = tagged_data['language_lookup']['es']
        print(f"\nLanguage: {es_info['name']}")
        print(f"Spoken in: {', '.join(es_info['spoken'])}")

    if verbose:
        with open('countries.json', 'r') as jf: print(jf.read())
        with open('tagged_country_data.json', 'r') as jf: print(jf.read())

    # From MIT App Inventor
    valid_languages = ["ar", "as", "bg", "bn", "brx", "bs", "ca", "cs", "cy", "da", "de", "doi", "el", "en", "es", "et", "fi", "fil", "fr", "gu", "he", "hi", "hr", "hu", "id", "is", "it", "ja", "jv", "km", "kn", "ko", "kok", "ks", "lt", "lv", "mai", "ml", "mni", "mr", "ms", "nb", "ne", "nl", "no", "or", "pa", "pl", "pt", "ro", "ru", "sa", "sat", "sd", "si", "sk", "sl", "sq", "sr", "su", "sv", "sw", "ta", "te", "th", "tr", "uk", "ur", "vi", "yue", "zh"]
    valid_countries = ["001", "150", "419", "ABW", "AGO", "AIA", "ALA", "ALB", "AND", "ARE", "ARG", "ASM", "ATG", "AUS", "AUT", "BDI", "BEL", "BEN", "BES", "BFA", "BGD", "BGR", "BHR", "BHS", "BIH", "BLM", "BLR", "BLZ", "BMU", "BOL", "BRA", "BRB", "BRN", "BWA", "CAF", "CAN", "CCK", "CHE", "CHL", "CHN", "CIV", "CMR", "COD", "COG", "COK", "COL", "COM", "CPV", "CRI", "CUB", "CUW", "CXR", "CYM", "CYP", "CZE", "DEU", "DGA", "DJI", "DMA", "DNK", "DOM", "DZA", "ECU", "EGY", "ERI", "ESH", "ESP", "EST", "FIN", "FJI", "FLK", "FRA", "FSM", "GAB", "GBR", "GEO", "GGY", "GHA", "GIB", "GIN", "GLP", "GMB", "GNB", "GNQ", "GRC", "GRD", "GRL", "GTM", "GUF", "GUM", "GUY", "HKG", "HND", "HRV", "HTI", "HUN", "IDN", "IMN", "IND", "IOT", "IRL", "IRQ", "ISL", "ISR", "ITA", "JAM", "JEY", "JOR", "JPN", "KAZ", "KEN", "KGZ", "KHM", "KIR", "KNA", "KOR", "KWT", "LBN", "LBR", "LBY", "LCA", "LIE", "LKA", "LSO", "LTU", "LUX", "LVA", "MAC", "MAF", "MAR", "MCO", "MDA", "MDG", "MDV", "MEX", "MHL", "MKD", "MLI", "MLT", "MNE", "MNP", "MOZ", "MRT", "MSR", "MTQ", "MUS", "MWI", "MYS", "MYT", "NAM", "NCL", "NER", "NFK", "NGA", "NIC", "NIU", "NLD", "NOR", "NPL", "NRU", "NZL", "OMN", "PAK", "PAN", "PCN", "PER", "PHL", "PLW", "PNG", "POL", "PRI", "PRK", "PRT", "PRY", "PSE", "PYF", "QAT", "REU", "ROU", "RUS", "RWA", "SAU", "SDN", "SEN", "SGP", "SHN", "SJM", "SLB", "SLE", "SLV", "SMR", "SOM", "SPM", "SRB", "SSD", "STP", "SUR", "SVK", "SVN", "SWE", "SWZ", "SXM", "SYC", "SYR", "TCA", "TCD", "TGO", "THA", "TKL", "TLS", "TON", "TTO", "TUN", "TUR", "TUV", "TWN", "TZA", "UGA", "UKR", "UMI", "URY", "USA", "VAT", "VCT", "VEN", "VGB", "VIR", "VNM", "VUT", "WLF", "WSM", "XEA", "XIC", "XKK", "YEM", "ZAF", "ZMB", "ZWE"]
    valid_short_countries = { tagged_data['country_lookup'].get(c, {'iso2': 'OTHER'})['iso2'] for c in valid_countries }
    print(f"\n# From MIT App Inventor: TTS languages, three-letter TTS countries, and two-letter TTS countries.")
    print('-', ','.join(valid_languages))
    print('-', ','.join(valid_countries))
    print('-', ','.join(sorted(valid_short_countries)))

    # For all valid_languages, collect 'spoken' list of tag in 'language_lookup'
    # that match tag with or without a '-XX' suffix.

    def valid_countries_for_language(language, languages,
        valid_countries=valid_countries):
        """Return a list of valid languages for a given country in countries by
        collecting the 'languages' tag for those whose 'code' is in valid_languages
        """
        countries = []
        for pl in languages:
            if pl == language or pl.startswith(f"{language}-"):
                countries += languages[pl]['spoken']
        return [ c for c in countries if c in valid_countries ]

    def find_language_name(language, languages=tagged_data['language_lookup']):
        """Check languages for name of language that matches exactly, or XX-YY."""
        if language in languages: return languages[language]['name']
        names = { languages[pl]['name'] for pl in languages
            if pl.startswith(f"{language}-") }
        assert len(names) <= 1, f"Too many names for '{language}': {names}"
        return list(names)[0] if len(names) == 1 else 'UNKNOWN'

    print(f"\nCountries for valid languages...")
    for vl in valid_languages:
        languages_lookup = tagged_data['language_lookup']
        name = find_language_name(vl, languages_lookup)
        countries = valid_countries_for_language(vl, languages_lookup)
        print(f"{vl:3s} {name[: 10]:10s}: {countries}")
        if len(countries) != len(set(countries)):
            print(f"Duplicate countries for {vl}: {countries}")

    def valid_languages_for_country(country, countries,
        valid_languages=valid_languages, valid_short_countries=valid_short_countries):
        """Return a list of valid languages for a given country in countries by
        collecting the 'languages' tag for those whose 'code' is in valid_languages
        or whose code is XX-YY and XX is in valid_languages and *also* YY is in
        valid_short_countries."""
        check = lambda n: n in valid_languages or (len(n) == 5
            and n[:2] in valid_languages and n[3:] in valid_short_countries)
        if country not in countries: return []
        # if re.sub(r'^(.{2})-(.*)', r'\1', d['code']) in valid_languages
        return [ d['code'] for d in countries[country]['languages']
            if check(d['code']) ]

    print(f"\nLanguages for valid country...")
    for vc in valid_countries:
        countries_lookup = tagged_data['country_lookup']
        name = countries_lookup[vc]['name'] if vc in countries_lookup else 'UNKNOWN'
        languages = valid_languages_for_country(vc, countries_lookup)
        print(f"{vc:3s} {name[: 10]:10s}: "
            f"{languages}")


Created iso-639-languages.json with 8130 language names
Created iso-3166-countries.json with 252 countries
Created unsd-m49-categories.json with 5 regions, 17 sub-regions, 7 intermediate-regions.
Created iso-data.json with 252 country keys, 508 language keys.
Processed 252 countries and 8130 language names.

languages_dict
aa: Afar
ab: Abkhazian
ae: Avestan
af: Afrikaans
ak: Akan
am: Amharic
an: Aragonese
ar: Arabic
as: Assamese
av: Avaric
ay: Aymara
az: Azerbaijani
ba: Bashkir
be: Belarusian
bg: Bulgarian
bh: Bihari languages
bi: Bislama
bm: Bambara
bn: Bengali
bo: Tibetan

countries_data
{'iso2': 'AD', 'iso3': 'AND', 'numeric_m49': '020', 'country': 'Andorra', 'continent': 'EU', 'tld': '.ad', 'languages': ['ca']}
{'iso2': 'AE', 'iso3': 'ARE', 'numeric_m49': '784', 'country': 'United Arab Emirates', 'continent': 'AS', 'tld': '.ae', 'languages': ['ar-AE', 'fa', 'en', 'hi', 'ur']}
{'iso2': 'AF', 'iso3': 'AFG', 'numeric_m49': '004', 'country': 'Afghanistan', 'continent': 'AS', 'tld': '.a